# 01 - Exploración de la API de Sofascore

Este notebook sirve para descubrir:
1. Qué endpoints responden sin bloqueos (403)
2. Cuál es el `seasonId` real del Mundial 2026
3. Cómo se estructuran los JSON de partidos, estadísticas e incidentes

**URL base confirmada:** `https://api.sofascore.com/api/v1`

**Path de torneos:** `/unique-tournament/{id}/...` (no `/tournament/`).

## 1. Setup e imports

In [ ]:
import json
import time

# Estrategia 1: requests puro (probablemente dé 403)
import requests

# Estrategia 2: curl_cffi (impersona TLS/headers de Chrome)
try:
    from curl_cffi import requests as curl_requests
except ImportError:
    curl_requests = None
    print("curl_cffi no instalado. Ejecuta: pip install curl_cffi")

BASE_URL = "https://api.sofascore.com/api/v1"
UNIQUE_TOURNAMENT_ID = 16  # Mundial

## 2. Prueba 1: requests puro (mínimo)

Sofascore suele bloquear requests sin headers realistas. Si esto da **403**, pasamos a la siguiente estrategia.

In [ ]:
url_seasons = f"{BASE_URL}/unique-tournament/{UNIQUE_TOURNAMENT_ID}/seasons"
print(f"Probando: {url_seasons}")

response = requests.get(url_seasons, timeout=30)
print(f"Status: {response.status_code}")
print(response.text[:500])

## 3. Prueba 2: requests con headers realistas

Copia tu `User-Agent` exacto desde el navegador (DevTools > Network > cualquier petición a Sofascore).

In [ ]:
headers_real = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36",
    "Accept": "*/*",
    "Accept-Language": "es-ES,es;q=0.9",
    "Referer": "https://www.sofascore.com/",
}

response = requests.get(url_seasons, headers=headers_real, timeout=30)
print(f"Status: {response.status_code}")
print(response.text[:500])

## 4. Prueba 3: curl_cffi (impersonación de Chrome)

Si requests sigue dando 403, `curl_cffi` imita el fingerprint TLS de un navegador real.
Instalación: `pip install curl_cffi`

In [ ]:
if curl_requests is not None:
    response = curl_requests.get(url_seasons, impersonate="chrome", timeout=30)
    print(f"Status: {response.status_code}")
    if response.status_code == 200:
        data = response.json()
        print(json.dumps(data, indent=2)[:2000])
    else:
        print(response.text[:500])
else:
    print("curl_cffi no está instalado. Salteando...")

## 5. Extraer seasonId del Mundial 2026

Una vez que obtengas 200 en la celda anterior, busca el `id` cuyo `name` o `year` sea "2026".

In [ ]:
# Ejecutar SOLO si la celda anterior dio 200
if response.status_code == 200:
    seasons = response.json().get("seasons", [])
    for s in seasons:
        print(f"ID: {s.get('id')} | Nombre: {s.get('name')} | Año: {s.get('year')}")

    # Anota aquí el seasonId del Mundial 2026:
    SEASON_ID = None  # <-- REEMPLAZA con el ID real
else:
    print("No se obtuvo 200. Resuelve el acceso primero.")

## 6. Explorar partidos por ronda

Sofascore organiza los partidos por ronda (`/events/round/{roundNumber}`).
Para el Mundial: rondas 1-N (fase de grupos), luego octavos, cuartos, semis, final.

Reemplaza `SEASON_ID` con el valor real que encontraste arriba.

In [ ]:
SEASON_ID = None  # <-- REEMPLAZA antes de ejecutar

if SEASON_ID is None:
    print("ERROR: Define SEASON_ID primero en la celda anterior.")
else:
    round_number = 1
    url_events = f"{BASE_URL}/unique-tournament/{UNIQUE_TOURNAMENT_ID}/season/{SEASON_ID}/events/round/{round_number}"
    print(f"Probando: {url_events}")

    if curl_requests is not None:
        r = curl_requests.get(url_events, impersonate="chrome", timeout=30)
    else:
        r = requests.get(url_events, headers=headers_real, timeout=30)

    print(f"Status: {r.status_code}")
    if r.status_code == 200:
        data = r.json()
        print("Keys:", list(data.keys()))
        events = data.get("events", [])
        print(f"Partidos en ronda {round_number}: {len(events)}")
        if events:
            print("\nPrimer partido:")
            print(json.dumps(events[0], indent=2)[:1500])
        
        # Guardar muestra para análisis offline
        with open("events_round_1_sample.json", "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print("\nGuardado en events_round_1_sample.json")

## 7. Explorar estadísticas de un partido

Toma un `event_id` de un partido finalizado del paso anterior y prueba estadísticas.

In [ ]:
EVENT_ID = None  # <-- REEMPLAZA con un ID real de un partido finalizado

if EVENT_ID is None:
    print("ERROR: Define EVENT_ID primero.")
else:
    url_stats = f"{BASE_URL}/event/{EVENT_ID}/statistics"
    print(f"Probando: {url_stats}")

    if curl_requests is not None:
        r = curl_requests.get(url_stats, impersonate="chrome", timeout=30)
    else:
        r = requests.get(url_stats, headers=headers_real, timeout=30)

    print(f"Status: {r.status_code}")
    if r.status_code == 200:
        data = r.json()
        with open(f"stats_{EVENT_ID}.json", "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"Guardado en stats_{EVENT_ID}.json")
        print("Keys de nivel superior:", list(data.keys()))
        print(json.dumps(data, indent=2)[:2000])

## 8. Explorar incidentes de un partido

Tarjetas, goles, sustituciones.

In [ ]:
if EVENT_ID is None:
    print("ERROR: Define EVENT_ID primero.")
else:
    url_incidents = f"{BASE_URL}/event/{EVENT_ID}/incidents"
    print(f"Probando: {url_incidents}")

    if curl_requests is not None:
        r = curl_requests.get(url_incidents, impersonate="chrome", timeout=30)
    else:
        r = requests.get(url_incidents, headers=headers_real, timeout=30)

    print(f"Status: {r.status_code}")
    if r.status_code == 200:
        data = r.json()
        with open(f"incidents_{EVENT_ID}.json", "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"Guardado en incidents_{EVENT_ID}.json")
        print(json.dumps(data, indent=2)[:2000])

## 9. Notas y descubrimientos

Anota aquí manualmente qué descubriste para documentar el pipeline:

- **¿Qué estrategia funcionó?** (requests / curl_cffi / otra)
- **¿Se necesitó cookie `__cf_bm`?**
- **¿El `seasonId` del Mundial 2026 es?**
- **¿Cuántas rondas tiene el torneo?**
- **¿La paginación usa `hasNextPage`?**
- **¿Dónde están posesión, tiros, xG en el JSON de estadísticas?**

---
## Anotaciones manuales (edita esta celda)

- Estrategia que funcionó: ...
- Season ID Mundial 2026: ...
- Rondas totales: ...
- Estructura de estadísticas: ...
- Otros hallazgos: ...